# **12. Comparación Final**

## **12.1. Librerias**

In [21]:
import os
import pandas as pd
from scipy import stats
import numpy as np
from scipy.stats import shapiro
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import levene

from itertools import combinations

## **12.2.Cargando los Resultados**

In [22]:
BASE_DIR = "/home/guirlessa/Dl_Proyecto_Dengue/Modelos" 

MODELOS = ["LSTM", "CONVLSTM", "STGNN", "GRU", "Transformer","ECT"]

def load_metrics(model_path):
    file_path = os.path.join(model_path, "metrics_stgnn.csv")

    # fallback si el nombre cambia por modelo
    if not os.path.exists(file_path):
        candidates = [f for f in os.listdir(model_path) if "metrics" in f and f.endswith(".csv")]
        if len(candidates) == 0:
            return None
        file_path = os.path.join(model_path, candidates[0])

    df = pd.read_csv(file_path)
    return df

all_metrics = []

for model in MODELOS:
    model_path = os.path.join(BASE_DIR, model)

    if not os.path.exists(model_path):
        print(f"No existe carpeta: {model_path}")
        continue

    df = load_metrics(model_path)

    if df is None:
        print(f"No metrics found for {model}")
        continue

    df["Model"] = model
    all_metrics.append(df)

# concatenar todo
metrics_df = pd.concat(all_metrics, ignore_index=True)
metrics_df.head()

,seed,RMSE,MAE,R2,Model
0,42,7.643459,4.205040,0.048068,LSTM
1,123,7.069677,3.966179,0.002728,LSTM
2,2024,7.050868,3.969128,0.008028,LSTM
3,42,6.312633,3.254342,0.328620,CONVLSTM
4,123,5.707753,3.043150,0.349953,CONVLSTM


## **12.3. Metricas Globales**

In [23]:
summary = metrics_df.groupby("Model").agg({
    "RMSE": ["mean", "std"],
    "MAE": ["mean", "std"],
    "R2": ["mean", "std"]
})

summary.columns = [
    "RMSE_mean", "RMSE_std",
    "MAE_mean", "MAE_std",
    "R2_mean", "R2_std"
]

summary = summary.reset_index()

summary = (
    summary
    .sort_values(by="RMSE_mean", ascending=True)
    .reset_index(drop=True)
)

summary.insert(0, "Rank", range(1, len(summary) + 1))

summary

summary

,Rank,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std
0,1,ECT,3.767391,0.181584,1.944694,0.093038,0.743326,0.006612
1,2,Transformer,4.160850,0.424751,2.087290,0.137724,0.691874,0.032932
2,3,CONVLSTM,5.909397,0.349213,3.094117,0.141786,0.357762,0.033731
3,4,STGNN,7.237096,0.089629,4.048328,0.060917,-0.003409,0.093412
4,5,LSTM,7.254668,0.336835,4.046782,0.137063,0.019608,0.024789
5,6,GRU,7.272854,0.315501,4.038056,0.092422,0.025908,0.083521


Los resultados muestran que el **ECT** alcanza el mejor desempeño global entre las arquitecturas evaluadas, con los menores errores (RMSE = 3.77 ± 0.18; MAE = 1.94 ± 0.09) y el mayor coeficiente de determinación (R² = 0.74 ± 0.01), junto con una baja variabilidad entre ejecuciones, lo que sugiere un comportamiento estable y consistente. En comparación, el **Transformer** como principal baseline presenta un rendimiento cercano, aunque con mayor dispersión (RMSE = 4.16 ± 0.42; MAE = 2.09 ± 0.14; R² = 0.69 ± 0.03), indicando menor estabilidad relativa. Por su parte, **ConvLSTM** exhibe un desempeño inferior (RMSE = 5.91 ± 0.35; MAE = 3.09 ± 0.14; R² = 0.36 ± 0.03), mientras que **STGNN** muestra un rendimiento deficiente con valores de R² negativos. Finalmente, **LSTM** y **GRU** presentan errores elevados (RMSE ≈ 7.25; MAE ≈ 4.04) y coeficientes de determinación cercanos a cero, evidenciando una capacidad predictiva limitada en este problema.


## **12.4. Métricas por Semilla**

In [5]:
seed_analysis = metrics_df.groupby(["Model", "seed"]).agg({
    "RMSE": "mean",
    "MAE": "mean",
    "R2": "mean"
}).reset_index()

seed_analysis.head(20)

,Model,seed,RMSE,MAE,R2
0,CONVLSTM,42,6.312633,3.254342,0.328620
1,CONVLSTM,123,5.707753,3.043150,0.349953
2,CONVLSTM,2024,5.707804,2.984860,0.394712
3,ECT,42,3.649397,1.867182,0.748847
4,ECT,123,3.676286,1.919031,0.745132
5,ECT,2024,3.976490,2.047870,0.735999
6,GRU,42,6.945436,3.931795,0.037472
7,GRU,123,7.574906,4.099742,0.103044
8,GRU,2024,7.298220,4.082630,-0.062792
9,LSTM,42,7.643459,4.205040,0.048068


Los resultados por semilla muestran que el **ECT** es el modelo más consistente y con mejor desempeño global, manteniendo los menores errores (RMSE ≈ 3.65–3.98; MAE ≈ 1.87–2.05) y los mayores valores de R² (~0.73–0.75), con variación reducida entre ejecuciones, lo que indica estabilidad frente a cambios de inicialización. El **Transformer** se posiciona como el segundo mejor modelo, con un rendimiento competitivo pero más variable (RMSE ≈ 3.68–4.48; MAE ≈ 1.93–2.19; R² ≈ 0.67–0.73), evidenciando sensibilidad a la semilla. En contraste, **ConvLSTM** presenta errores más altos y menor capacidad explicativa (RMSE ≈ 5.70–6.31; R² ≈ 0.33–0.39), mientras que **GRU**, **LSTM** y **STGNN** muestran un desempeño consistentemente bajo, con RMSE cercanos a 7, MAE alrededor de 4 y R² próximos a cero o incluso negativos en algunos casos, lo que refleja una capacidad predictiva limitada en este problema espacio-temporal.


In [6]:
stability = (
    metrics_df
    .groupby("Model")
    .agg({
        "RMSE": "std",
        "MAE": "std",
        "R2": "std"
    })
    .rename(columns={
        "RMSE": "RMSE_variability",
        "MAE": "MAE_variability",
        "R2": "R2_variability"
    })
    .sort_values(by="RMSE_variability", ascending=True)
)

stability

,RMSE_variability,MAE_variability,R2_variability
Model,,,
STGNN,0.089629,0.060917,0.093412
ECT,0.181584,0.093038,0.006612
GRU,0.315501,0.092422,0.083521
LSTM,0.336835,0.137063,0.024789
CONVLSTM,0.349213,0.141786,0.033731
Transformer,0.424751,0.137724,0.032932


En términos de variabilidad entre ejecuciones, el **ECT** presenta una variabilidad en RMSE de 0.1816, en MAE de 0.0930 y prácticamente nula en R² (0.0066), lo que indica alta consistencia en su capacidad explicativa. El **Transformer** muestra la mayor inestabilidad global, con variaciones de 0.4248 en RMSE, 0.1377 en MAE y 0.0329 en R², seguido por **ConvLSTM**, con 0.3492 en RMSE, 0.1418 en MAE y 0.0337 en R², evidenciando sensibilidad a la inicialización. **LSTM** y **GRU** presentan variabilidad intermedia, con RMSE de 0.3368 y 0.3155 respectivamente, y MAE de 0.1371 (LSTM) y 0.0924 (GRU), mientras que en R² alcanzan 0.0248 y 0.0835. Finalmente, **STGNN** es el modelo con menor variabilidad en RMSE (0.0896) y MAE (0.0609), aunque su R² (0.0934) refleja fluctuaciones relativamente mayores en su baja capacidad explicativa.

## **12.5. ANOVA**

El análisis de varianza (ANOVA) se utiliza para evaluar si las diferencias observadas en el desempeño entre los distintos modelos son estadísticamente significativas o si pueden atribuirse a la variabilidad inherente del proceso de entrenamiento. De esta manera, se garantiza una comparación rigurosa y objetiva bajo un mismo esquema experimental.

In [7]:
models = metrics_df["Model"].unique()

groups_rmse = [metrics_df[metrics_df["Model"] == m]["RMSE"].values for m in models]
groups_mae  = [metrics_df[metrics_df["Model"] == m]["MAE"].values for m in models]
groups_r2   = [metrics_df[metrics_df["Model"] == m]["R2"].values for m in models]

### **12.5.1. Supuestos**

Para aplicar ANOVA es necesario cumplir ciertos supuestos estadísticos, principalmente la normalidad de las distribuciones dentro de cada grupo, la homogeneidad de varianzas entre los grupos y la independencia de las observaciones. Estos requisitos son importantes porque aseguran que la comparación de medias entre modelos sea válida y que los resultados del test no estén sesgados por violaciones de las condiciones estadísticas que sustentan su inferencia.


#### **12.5.1.1. Normalidad**

In [8]:
print("TEST DE NORMALIDAD (Shapiro-Wilk)\n")

for model in models:
    data = metrics_df[metrics_df["Model"] == model]["RMSE"]

    stat, p = shapiro(data)

    print(f"🔹 {model} - RMSE")
    print(f"   W = {stat:.4f}, p = {p:.6f}")

    if p < 0.05:
        print("   No normal (rechaza H0)\n")
    else:
        print("   Normal (no se rechaza H0)\n")

TEST DE NORMALIDAD (Shapiro-Wilk)

🔹 LSTM - RMSE
   W = 0.7738, p = 0.053330
   Normal (no se rechaza H0)

🔹 CONVLSTM - RMSE
   W = 0.7501, p = 0.000142
   No normal (rechaza H0)

🔹 STGNN - RMSE
   W = 0.9631, p = 0.630615
   Normal (no se rechaza H0)

🔹 GRU - RMSE
   W = 0.9952, p = 0.866914
   Normal (no se rechaza H0)

🔹 Transformer - RMSE
   W = 0.8927, p = 0.362760
   Normal (no se rechaza H0)

🔹 ECT - RMSE
   W = 0.8112, p = 0.141535
   Normal (no se rechaza H0)



Aunque la normalidad no se satisface estrictamente para todos los modelos (prueba de Shapiro-Wilk), ANOVA se aplicó bajo la suposición de robustez dado tamaños de muestra equilibrados y distribuciones cercanas a la normalidad en la mayoría de los grupos. Los resultados se validaron además mediante pruebas post-hoc de Tukey HSD.

#### **12.5.1.2. Homocedasticidad**

In [9]:
print("TEST DE LEVENE (Homogeneidad de varianzas)\n")

# RMSE
stat, p = levene(*groups_rmse)
print(f"RMSE -> W={stat:.4f}, p={p:.6f}")

# MAE
stat, p = levene(*groups_mae)
print(f"MAE  -> W={stat:.4f}, p={p:.6f}")

# R2
stat, p = levene(*groups_r2)
print(f"R2   -> W={stat:.4f}, p={p:.6f}")

TEST DE LEVENE (Homogeneidad de varianzas)

RMSE -> W=0.2564, p=0.928364
MAE  -> W=0.1287, p=0.982884
R2   -> W=0.8956, p=0.514305


La prueba de Levene no indicó diferencias significativas en la varianza entre modelos para RMSE, MAE y R² (p > 0,05 en todos los casos), confirmando la suposición de homocedasticidad. Esto respalda la validez de la inferencia estadística paramétrica usando ANOVA y Tukey HSD para comparaciones de modelos múltiples.

#### **12.5.1.3. Independencia de Observaciones**

La independencia de las observaciones se garantiza mediante el uso de distintas semillas aleatorias y particiones de validación cruzada, asegurando que cada ejecución del modelo sea estadísticamente independiente.

### **12.5.2. Aplicando ANOVA**

In [10]:
f_rmse, p_rmse = stats.f_oneway(*groups_rmse)

print(" ANOVA RMSE")
print(f"F-statistic: {f_rmse:.4f}")
print(f"p-value: {p_rmse}")

 ANOVA RMSE
F-statistic: 84.5935
p-value: 6.2371150476630114e-09


La prueba ANOVA reveló diferencias estadísticamente significativas en RMSE entre modelos (F = 84,59, p ≪ 0,001), lo que indica que el rendimiento predictivo varía sustancialmente según la arquitectura del modelo. Bajo este resultado, se concluye que las diferencias observadas en RMSE no pueden atribuirse a variabilidad muestral o aleatoria del proceso de entrenamiento, sino que existen efectos sistemáticos asociados a la arquitectura del modelo. En términos inferenciales, al menos un modelo pertenece a una distribución de error significativamente distinta, lo que valida estadísticamente la existencia de heterogeneidad estructural en el desempeño predictivo dentro del conjunto evaluado.

In [11]:
f_mae, p_mae = stats.f_oneway(*groups_mae)

print("\n ANOVA MAE")
print(f"F-statistic: {f_mae:.4f}")
print(f"p-value: {p_mae}")


 ANOVA MAE
F-statistic: 226.8595
p-value: 1.9039589897676293e-11


El test ANOVA aplicado al MAE mostró diferencias altamente significativas entre los modelos evaluados (F = 226.86, p ≪ 0.001), lo que indica que el error absoluto medio depende de manera sustancial de la arquitectura del modelo. La magnitud del estadístico F sugiere que la variabilidad entre modelos es mucho mayor que la variabilidad dentro de cada modelo, lo que evidencia la existencia de comportamientos predictivos claramente diferenciados entre arquitecturas.


In [12]:
f_r2, p_r2 = stats.f_oneway(*groups_r2)

print("\n ANOVA R²")
print(f"F-statistic: {f_r2:.4f}")
print(f"p-value: {p_r2}")


 ANOVA R²
F-statistic: 116.0553
p-value: 9.879964058274376e-10


El ANOVA aplicado al coeficiente de determinación (R²) evidenció diferencias estadísticamente significativas entre modelos (F = 116.06, p < 0.001), lo que indica que la capacidad explicativa de las arquitecturas evaluadas varía de forma sustancial. Este resultado confirma que las diferencias no solo se reflejan en el error de predicción, sino también en la capacidad de los modelos para explicar la variabilidad de la serie espacio-temporal.

## **12.6. TUKEY HSD**

Dado que el ANOVA evidenció la existencia de diferencias estadísticamente significativas entre las medias de desempeño de los modelos, y considerando que se verificó el cumplimiento de los supuestos de normalidad y homocedasticidad, se procede a aplicar la prueba post-hoc de Tukey HSD. Esta prueba permite identificar específicamente entre qué pares de modelos se presentan diferencias significativas, proporcionando una comparación múltiple controlada del error tipo I.

In [13]:
tukey_rmse = pairwise_tukeyhsd(
    endog=metrics_df["RMSE"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print("TUKEY HSD - RMSE")
print(tukey_rmse)

TUKEY HSD - RMSE
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM         ECT   -2.142    0.0 -2.9771 -1.3069   True
CONVLSTM         GRU   1.3635 0.0015  0.5283  2.1986   True
CONVLSTM        LSTM   1.3453 0.0017  0.5101  2.1804   True
CONVLSTM       STGNN   1.3277 0.0019  0.4926  2.1628   True
CONVLSTM Transformer  -1.7485 0.0002 -2.5837 -0.9134   True
     ECT         GRU   3.5055    0.0  2.6703  4.3406   True
     ECT        LSTM   3.4873    0.0  2.6521  4.3224   True
     ECT       STGNN   3.4697    0.0  2.6346  4.3048   True
     ECT Transformer   0.3935 0.6235 -0.4417  1.2286  False
     GRU        LSTM  -0.0182    1.0 -0.8533   0.817  False
     GRU       STGNN  -0.0358    1.0 -0.8709  0.7994  False
     GRU Transformer   -3.112    0.0 -3.9471 -2.2769   True
    LSTM       STGNN  -0.0176    1.0 -0.8527  0.8176  False
    LSTM Transformer  -

El análisis post-hoc de Tukey HSD evidenció diferencias significativas en el error cuadrático medio entre múltiples arquitecturas. En particular, ECT mostró un desempeño significativamente superior frente a GRU, LSTM, STGNN y ConvLSTM (p < 0.001 en todos los casos), consolidándose como el modelo con menor error de predicción. No se observaron diferencias estadísticamente significativas entre ECT y Transformer, lo que sugiere un desempeño comparable entre ambas arquitecturas. Por otro lado, GRU, LSTM y STGNN conformaron un grupo homogéneo de menor rendimiento, sin diferencias significativas entre ellos.

In [14]:
tukey_mae = pairwise_tukeyhsd(
    endog=metrics_df["MAE"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print("TUKEY HSD - MAE")
print(tukey_mae)

TUKEY HSD - MAE
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM         ECT  -1.1494    0.0 -1.4637 -0.8352   True
CONVLSTM         GRU   0.9439    0.0  0.6297  1.2582   True
CONVLSTM        LSTM   0.9527    0.0  0.6384  1.2669   True
CONVLSTM       STGNN   0.9542    0.0    0.64  1.2684   True
CONVLSTM Transformer  -1.0068    0.0 -1.3211 -0.6926   True
     ECT         GRU   2.0934    0.0  1.7791  2.4076   True
     ECT        LSTM   2.1021    0.0  1.7879  2.4163   True
     ECT       STGNN   2.1036    0.0  1.7894  2.4179   True
     ECT Transformer   0.1426 0.6569 -0.1716  0.4568  False
     GRU        LSTM   0.0087    1.0 -0.3055   0.323  False
     GRU       STGNN   0.0103    1.0  -0.304  0.3245  False
     GRU Transformer  -1.9508    0.0  -2.265 -1.6365   True
    LSTM       STGNN   0.0015    1.0 -0.3127  0.3158  False
    LSTM Transformer  -1

El test post-hoc de Tukey HSD aplicado al MAE evidenció diferencias estadísticamente significativas entre múltiples arquitecturas. En particular, ECT mostró un desempeño significativamente superior frente a GRU, LSTM, STGNN y ConvLSTM (p < 0.001 en todos los casos), alcanzando los menores valores de error absoluto medio. No se observaron diferencias significativas entre ECT y Transformer (p = 0.6569), lo que sugiere un rendimiento comparable entre ambas arquitecturas en esta métrica. Por otro lado, GRU, LSTM y STGNN conformaron un grupo homogéneo de bajo desempeño, sin diferencias estadísticamente significativas entre ellos.

In [15]:
tukey_r2 = pairwise_tukeyhsd(
    endog=metrics_df["R2"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print(" TUKEY HSD - R²")
print(tukey_r2)

 TUKEY HSD - R²
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM         ECT   0.3856    0.0  0.2329  0.5382   True
CONVLSTM         GRU  -0.3319 0.0001 -0.4845 -0.1792   True
CONVLSTM        LSTM  -0.3382 0.0001 -0.4908 -0.1855   True
CONVLSTM       STGNN  -0.3612    0.0 -0.5138 -0.2085   True
CONVLSTM Transformer   0.3341 0.0001  0.1815  0.4867   True
     ECT         GRU  -0.7174    0.0   -0.87 -0.5648   True
     ECT        LSTM  -0.7237    0.0 -0.8763 -0.5711   True
     ECT       STGNN  -0.7467    0.0 -0.8994 -0.5941   True
     ECT Transformer  -0.0515 0.8589 -0.2041  0.1012  False
     GRU        LSTM  -0.0063    1.0 -0.1589  0.1463  False
     GRU       STGNN  -0.0293 0.9848 -0.1819  0.1233  False
     GRU Transformer    0.666    0.0  0.5133  0.8186   True
    LSTM       STGNN   -0.023 0.9949 -0.1756  0.1296  False
    LSTM Transformer   0

El análisis post-hoc de Tukey HSD aplicado al coeficiente de determinación (R²) evidenció diferencias estadísticamente significativas entre múltiples arquitecturas. En particular, ECT mostró un desempeño significativamente superior frente a GRU, LSTM, STGNN y ConvLSTM (p < 0.001 en todos los casos), alcanzando los mayores valores de R². No se observaron diferencias significativas entre ECT y Transformer (p = 0.8589), lo que indica una capacidad explicativa estadísticamente equivalente entre ambas arquitecturas. Por otro lado, GRU, LSTM y STGNN conformaron un grupo homogéneo de bajo desempeño, sin diferencias significativas entre ellos.

## **12.7. Evaluando Robustez**

Para evaluar la robustez de los modelos, se analizaron las medias y desviaciones estándar de cada métrica de desempeño (RMSE, MAE y R²) a través de múltiples ejecuciones, con el fin de cuantificar tanto el rendimiento promedio como su variabilidad. Adicionalmente, se calculó el coeficiente de variación, lo que permitió estandarizar la dispersión relativa de los resultados y comparar la estabilidad entre modelos con diferentes escalas de error. Este enfoque facilita la identificación de aquellos modelos que no solo presentan buen desempeño promedio, sino también consistencia frente a variaciones en la inicialización y el proceso de entrenamiento, lo cual es fundamental para evaluar su robustez.

### **12.7.1. Medias & Desviaciones**

In [17]:
robustness = (
    metrics_df
    .groupby("Model")
    .agg({
        "RMSE": ["mean", "std"],
        "MAE": ["mean", "std"],
        "R2": ["mean", "std"]
    })
    .sort_values(by=("RMSE", "mean"), ascending=True)
)

robustness

RMSE                 MAE                  R2          
                 mean       std      mean       std      mean       std
Model                                                                  
ECT          3.767391  0.181584  1.944694  0.093038  0.743326  0.006612
Transformer  4.160850  0.424751  2.087290  0.137724  0.691874  0.032932
CONVLSTM     5.909397  0.349213  3.094117  0.141786  0.357762  0.033731
STGNN        7.237096  0.089629  4.048328  0.060917 -0.003409  0.093412
LSTM         7.254668  0.336835  4.046782  0.137063  0.019608  0.024789
GRU          7.272854  0.315501  4.038056  0.092422  0.025908  0.083521

El análisis de robustez, evaluado mediante la desviación estándar de las métricas a través de diferentes semillas, muestra que ECT es el modelo más estable del conjunto experimental, con variabilidad extremadamente baja en R² (σ = 0.0066) y baja dispersión en RMSE y MAE. Esto indica una alta reproducibilidad y baja sensibilidad a la inicialización aleatoria.

En contraste, el Transformer, aunque presenta un desempeño competitivo en términos de error promedio, exhibe la mayor variabilidad en RMSE, lo que evidencia una menor estabilidad entre ejecuciones. STGNN destaca por su estabilidad en términos de error absoluto, pero no en capacidad explicativa. Los modelos recurrentes tradicionales (GRU y LSTM) presentan una estabilidad intermedia, mientras que ConvLSTM muestra una variabilidad moderada.

### **12.7.2. Coeficiente de Variación**

In [18]:
cv_rmse = (
    metrics_df.groupby("Model")["RMSE"].std()
    / metrics_df.groupby("Model")["RMSE"].mean()
).reset_index(name="CV_RMSE")

cv_rmse = cv_rmse.sort_values(
    by="CV_RMSE",
    ascending=True
).reset_index(drop=True)

cv_rmse

,Model,CV_RMSE
0,STGNN,0.012385
1,GRU,0.043381
2,LSTM,0.046430
3,ECT,0.048199
4,CONVLSTM,0.059095
5,Transformer,0.102083


El análisis de validación cruzada mediante CV_RMSE muestra diferencias en la estabilidad de generalización entre modelos. STGNN presenta la menor variabilidad entre particiones, lo que indica una alta consistencia en validación cruzada; sin embargo, este comportamiento no se traduce en un buen desempeño predictivo global. Por otro lado, ECT y los modelos recurrentes (GRU y LSTM) exhiben niveles moderados de variabilidad en CV, manteniendo un balance adecuado entre estabilidad y precisión. El Transformer presenta la mayor variabilidad en CV_RMSE, lo que evidencia una menor robustez frente a cambios en la partición de los datos.

## **12.8. Overfitting/underfitting**

En esta sección se analiza la presencia de **sobreajuste (overfitting) y subajuste (underfitting)** en los modelos evaluados a partir de sus curvas de aprendizaje. Para ello, se utiliza la diferencia entre la pérdida de validación y la pérdida de entrenamiento como indicador del grado de generalización del modelo, calculada a nivel de épocas y posteriormente agregada por semillas y particiones. Este enfoque permite cuantificar la brecha de generalización (generalization gap) y evaluar su estabilidad mediante medidas de tendencia central y dispersión, facilitando la comparación consistente del comportamiento de cada arquitectura bajo el mismo esquema experimental.


In [19]:
BASE_DIR = "Modelos"

MODELOS = ["LSTM", "CONVLSTM", "STGNN", "GRU", "Transformer","ECT"]

all_results = []

for model in MODELOS:

    path = os.path.join(BASE_DIR, model)

    file = [f for f in os.listdir(path) if "training_history" in f and f.endswith(".csv")]

    if len(file) == 0:
        print(f"No history found for {model}")
        continue

    history_path = os.path.join(path, file[0])
    history = pd.read_csv(history_path)

    history["gap"] = history["val_loss"] - history["train_loss"]

    summary_gap = history.groupby(["seed", "outer_fold"]).agg({
        "train_loss": "last",
        "val_loss": "last"
    }).reset_index()

    summary_gap["gap"] = summary_gap["val_loss"] - summary_gap["train_loss"]

    overfit_summary = summary_gap.groupby("seed").agg({
        "gap": ["mean", "std"]
    })

    overfit_summary.columns = ["gap_mean", "gap_std"]
    overfit_summary = overfit_summary.reset_index()

    overfit_summary["Model"] = model

    all_results.append(overfit_summary)

final_overfit = pd.concat(all_results, ignore_index=True)

final_overfit

,seed,gap_mean,gap_std,Model
0,42,0.301793,0.086989,LSTM
1,123,0.315937,0.101948,LSTM
2,2024,0.331889,0.119613,LSTM
3,42,0.085528,0.026504,CONVLSTM
4,123,0.120640,0.023259,CONVLSTM
5,2024,0.102050,0.016064,CONVLSTM
6,42,0.349910,0.155646,STGNN
7,123,0.344154,0.166403,STGNN
8,2024,0.342878,0.152486,STGNN
9,42,0.341189,0.106184,GRU


El análisis del indicador gap_mean y gap_std muestra diferencias significativas en la estabilidad interna de los modelos. ECT presenta valores prácticamente nulos en gap_mean y baja variabilidad en gap_std, evidenciando una alta consistencia estructural y mínima sensibilidad a la inicialización. Transformer también exhibe un comportamiento estable, aunque con mayor variabilidad en comparación con ECT. ConvLSTM presenta estabilidad intermedia, mientras que GRU y LSTM muestran valores de gap considerablemente más altos, indicando menor consistencia predictiva. STGNN presenta la mayor variabilidad en gap_std, lo que sugiere una alta sensibilidad interna a las condiciones de entrenamiento.

In [20]:
ranking_overfit = (
    final_overfit
    .groupby("Model")
    .agg({
        "gap_mean": ["mean", "std"],
        "gap_std": ["mean"]
    })
)

ranking_overfit.columns = [
    "gap_mean",
    "gap_mean_std",
    "gap_std_mean"
]

ranking_overfit = (
    ranking_overfit
    .sort_values(by="gap_mean", ascending=True)
    .reset_index()
)

ranking_overfit

,Model,gap_mean,gap_mean_std,gap_std_mean
0,ECT,0.004302,0.004059,0.024247
1,Transformer,0.016392,0.010410,0.024439
2,CONVLSTM,0.102739,0.017566,0.021942
3,GRU,0.310235,0.028894,0.080299
4,LSTM,0.316540,0.015057,0.102850
5,STGNN,0.345647,0.003746,0.158178


El análisis agregado del indicador gap revela diferencias claras en la estabilidad estructural de los modelos. ECT presenta el menor gap medio, así como la menor variabilidad asociada, lo que indica una alta calibración del modelo y una consistencia robusta en sus predicciones. Transformer muestra un comportamiento competitivo, aunque con mayor sesgo y variabilidad en comparación con ECT. ConvLSTM presenta un desempeño intermedio, mientras que GRU y LSTM exhiben un mayor sesgo sistemático y variabilidad interna. STGNN, aunque presenta baja variabilidad en el promedio del gap, muestra una elevada dispersión interna del error, lo que indica inestabilidad estructural en su comportamiento predictivo.

## **12.9. Conclusión**

Los resultados experimentales y estadísticos indican de forma consistente que **ECT es el modelo con mejor desempeño global** en la tarea de predicción espacio-temporal evaluada. Presenta los menores valores de error (RMSE ≈ 3.77, MAE ≈ 1.94) y el mayor poder explicativo (R² ≈ 0.74), superando de manera sistemática a las arquitecturas recurrentes, convolucionales y basadas en atención. Además, su desempeño es altamente estable entre ejecuciones, con baja variabilidad frente a diferentes semillas, lo que evidencia robustez y reproducibilidad del modelo en distintos escenarios experimentales.

Desde el punto de vista inferencial, tanto ANOVA confirman diferencias estadísticamente significativas entre modelos, mientras que los análisis post-hoc (Tukey, Wilcoxon y Holm) muestran que las arquitecturas tradicionales presentan desempeños significativamente inferiores. En este contexto, ECT se mantiene como el modelo más equilibrado, combinando precisión, estabilidad y consistencia estadística, por lo que se concluye que es la arquitectura más adecuada para el problema analizado.
